This is a continuation of my first icl_measure notebook. Just got tired of scrolling.

Here's the plan of action: 

* Create masked 55-66 arrays in gri. THROUGHOUT THE NOTEBOOK IT'S ASSUMED I'M USING THE _CLIPS COADDS
* Determine the background values in each band to subtract from mean intensities.
* Generate g-r, r-i, and g-i (important for NSF proposal) color plots 

In [ ]:
from astropy.io import fits
import pandas as pd
import numpy as np
import numpy.ma as ma
import matplotlib.pyplot as plt
import astropy
from astropy.visualization import ManualInterval
import photutils
from astropy.visualization import AsinhStretch
from astropy.visualization import ZScaleInterval
from photutils.aperture import EllipticalAperture, EllipticalAnnulus
from photutils.centroids import (centroid_1dg, centroid_com)
from photutils.aperture import ApertureStats
from photutils.isophote import Ellipse
from photutils.isophote import EllipseGeometry
from matplotlib.patches import Ellipse as mpl_Ellipse
plt.rcParams['figure.dpi'] = 1000

In [ ]:
# Useful functions

# by default we clip and stretch the data before displaying it
def display(mat, vmin, vmax, tag):
    interval = ManualInterval(vmin, vmax)
    mat_1 = interval(mat)
    stretch = AsinhStretch(0.0004)
    mat_2 = stretch(mat_1)

    return mat_2

def display_zscale(mat):
    z1, z2 = ZScaleInterval().get_limits(mat)

    return z1,z2

def get_masked_coadd(band_list):

    masked_coadd_list = []

    for band in band_list:
        hdul_coadd = fits.open(f'{coadd_dir}{cln}_{band}{coadd_fits_filename}')
        hdul_mask = fits.open(f'{mask_dir}{cln}_{band}{mask_fits_filename}')
        data_coadd = hdul_coadd[0].data
        data_mask = hdul_mask[0].data
        masked_coadd = ma.masked_array(data_coadd, mask=data_mask)
        masked_coadd_list.append(masked_coadd)
        # z1, z2 = display_zscale(masked_coadd)
        # plt.figure()
        # plt.imshow(masked_coadd, vmin=z1, vmax=z2)
        # plt.gca().invert_yaxis()
        # plt.title(f'Masked Coadd {band} Zscale Skycorr')
        # plt.savefig(f"masked_coadd_zscale_skycorr_{band}.png", dpi = 4096)

    return masked_coadd_list

def mask_single_coadd(coadd_arr, mask_arr, bcg_val):
    arr = np.where(mask_arr == bcg_val,0,mask_arr)
    arr = np.where(arr != 0,1, arr)
    
    return ma.masked_array(coadd_arr, mask=arr)

def fit_ellip_annuli(band, coadd, center, spacings, theta=45):
    # Array to hold summed intensities in each annulus
    sums = []
    # Array to hold annulus areas
    areas = []

    # Plot coadd
    plt.figure(figsize=(10,8))
    plt.imshow(display(coadd, -0.1, 250, 'fig1')*255)
    plt.plot(center[0],center[1], 'r+')

    # Begin fitting annuli
    for ind in range(len(spacings) - 1):
        annulus = EllipticalAnnulus(center, a_in=spacings[ind], a_out=spacings[ind+1], \
                               b_out=spacings[ind], theta=theta)
        annulus.plot(color='red', lw=1)
        sums.append(list(annulus.do_photometry(coadd)[0]))
        # aperstats = ApertureStats(data, annulus_aperture)
        areas.append(annulus.area)

    # Convert to list
    sums = sum(sums, [])

    # Other plot parameters
    plt.gca().invert_yaxis()
    plt.colorbar(label="Intensity", orientation="vertical") 
    plt.xlabel('x (pixels)')
    plt.ylabel('y (pixels)')
    plt.title(f'{band} Band Elliptical Annuli')
    plt.show()

    # Return annuli sums (units of counts) and areas (untis of pixels)
    return sums, areas
    

In [ ]:
# Directory information

# CLUSTER NAME
cln = 'A85'

mask_dir = '~/data/zescalan_research/A85_testing/SExtractor/'
coadd_dir = '~/data/zescalan_research/A85_testing/'

mask_g_filename = 'seg_g_clips.fits'
mask_r_filename = 'seg_r_clips.fits'
mask_i_filename = 'seg_i_clips.fits'

coadd_filename = '55-66_deepCoadd_clips.fits'

# Open data
mask_g_hdul = fits.open(mask_dir + mask_g_filename)
mask_r_hdul = fits.open(mask_dir + mask_r_filename)
mask_i_hdul = fits.open(mask_dir + mask_i_filename)

coadd_g_hdul = fits.open(f'{coadd_dir}{cln}_g{coadd_filename}')
coadd_r_hdul = fits.open(f'{coadd_dir}{cln}_r{coadd_filename}')
coadd_i_hdul = fits.open(f'{coadd_dir}{cln}_i{coadd_filename}')

# (8000,8000) arrays
coadd_g_data = coadd_g_hdul[0].data
coadd_r_data = coadd_r_hdul[0].data
coadd_i_data = coadd_i_hdul[0].data

mask_g_data = mask_g_hdul[0].data
mask_r_data = mask_r_hdul[0].data
mask_i_data = mask_i_hdul[0].data

In [ ]:
# Apply masks to all sources except the BCG
bcg_unmsk_g_data = mask_single_coadd(coadd_g_data, mask_g_data, 13946)
bcg_unmsk_r_data = mask_single_coadd(coadd_r_data, mask_r_data, 26170)
bcg_unmsk_i_data = mask_single_coadd(coadd_i_data, mask_i_data, 16265)

In [ ]:
rough_center = np.array([4050, 3750])

# In case I want to see the masked coadds
# plt.figure()
# plt.imshow(display(bcg_unmsk_g_data, -0.1, 250, 'fig1')*255)
# plt.plot(rough_center[0],rough_center[1], 'r+')
# plt.gca().invert_yaxis()
# plt.colorbar(label="Intensity", orientation="vertical") 
# plt.xlabel('x (pixels)')
# plt.ylabel('y (pixels)')
# plt.title('g Coadd BCG Unmasked')
# plt.show()

# plt.figure()
# plt.imshow(display(bcg_unmsk_r_data, -0.1, 250, 'fig1')*255)
# plt.plot(rough_center[0],rough_center[1], 'r+')
# plt.gca().invert_yaxis()
# plt.colorbar(label="Intensity", orientation="vertical") 
# plt.xlabel('x (pixels)')
# plt.ylabel('y (pixels)')
# plt.title('r Coadd BCG Unmasked')
# plt.show()

# plt.figure()
# plt.imshow(display(bcg_unmsk_i_data, -0.1, 250, 'fig1')*255)
# plt.plot(rough_center[0],rough_center[1], 'r+')
# plt.gca().invert_yaxis()
# plt.colorbar(label="Intensity", orientation="vertical") 
# plt.xlabel('x (pixels)')
# plt.ylabel('y (pixels)')
# plt.title('i Coadd BCG Unmasked')
# plt.show()

In [ ]:
# Place annuli around each band masked coadd and get background values 

# Just using rough center to fit. I don't think it matters too much...
# I'll also use the same logarithmic spacings
# log_spacings = np.logspace(3,3.6,6) This fits outer reaches of field
log_spacings = np.logspace(1,3.6,15) #This includes BCG/ICL
sums_g, areas_g = fit_ellip_annuli('g', bcg_unmsk_g_data, rough_center, log_spacings, 45)
sums_r, areas_r = fit_ellip_annuli('r', bcg_unmsk_r_data, rough_center, log_spacings, 45)
sums_i, areas_i = fit_ellip_annuli('i', bcg_unmsk_i_data, rough_center, log_spacings, 45)

In [ ]:
# Convert to mean intensities and units of arcsecs
# mean = total intensity/area

means_g_bkg = [a/b for a,b in zip(sums_g,areas_g)]
means_r_bkg = [a/b for a,b in zip(sums_r,areas_r)]
means_i_bkg = [a/b for a,b in zip(sums_i,areas_i)]

print(means_g_bkg); print(means_r_bkg) ;print(means_i_bkg)

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(log_spacings[1:], means_g_bkg, s = 50, color= 'g', marker ='+', label='g')
plt.plot(log_spacings[1:], means_g_bkg, 'g')
plt.scatter(log_spacings[1:], means_r_bkg, s = 50, color= 'r', marker ='o', label='r')
plt.plot(log_spacings[1:], means_r_bkg, 'r')
plt.scatter(log_spacings[1:], means_i_bkg, s = 50, color= 'k', marker ='s', label='i')
plt.plot(log_spacings[1:], means_i_bkg, 'k')
# plt.gca().invert_yaxis()
plt.xscale('log')
plt.legend()
plt.xlabel('SMA (pix)')
plt.ylabel('Mean Intensity (Counts/pix)')
plt.title('Background Intensities')
plt.show()

plt.figure(figsize=(6,4))
plt.scatter(log_spacings[1:], means_g_bkg, s = 50, color= 'g', marker ='+', label='g')
plt.plot(log_spacings[1:], means_g_bkg, 'g')
plt.scatter(log_spacings[1:], means_r_bkg, s = 50, color= 'r', marker ='o', label='r')
plt.plot(log_spacings[1:], means_r_bkg, 'r')
plt.scatter(log_spacings[1:], means_i_bkg, s = 50, color= 'k', marker ='s', label='i')
plt.plot(log_spacings[1:], means_i_bkg, 'k')
plt.xscale('log')
plt.yscale('log')
plt.xlim((4e2,5e3))
plt.legend()
plt.xlabel('SMA (pix)')
plt.ylabel('Mean Intensity (Counts/pix)')
plt.title('Background Intensities')
plt.show()

g and r curves seem to level off, but i increases? There is a "noticeable" increase in intensities in bottom-middle of coadd (see above)

In [ ]:
# Averaging the final three points to get the background levels
bkg_g_cperpix = np.mean(means_g_bkg[-3:])
bkg_r_cperpix = np.mean(means_r_bkg[-3:])
bkg_i_cperpix = np.mean(means_i_bkg[-3:])

print(means_g_bkg)
print(means_g_bkg - bkg_g_cperpix)